In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

# 1. Load Dataset
data_path = '../data/raw/PaySim_log.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("\nTransaction Types Distribution:\n", df['type'].value_counts())

# In PaySim fraud happens in TRANSFER and CASH_OUT types only
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()


# 2. Feature Engineering (Domain-specific for PaySim)
# Calculating Balance change errors
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# One-Hot Encoding for 'type' (TRANSFER, CASH_OUT)
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# Drop non-numeric IDs and target identifiers
df.drop(['nameOrig', 'nameDest', 'isFlaggedFraud'], axis=1, inplace=True, errors='ignore')

# 3. Scaling 'amount', 'oldbalanceOrg', etc.
scaler = RobustScaler()
cols_to_scale = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("\nProcessed Shape (Filtered TRANSFER & CASH_OUT):", df.shape)
print("Fraud Distribution:\n", df['isFraud'].value_counts())
print("\nHead:\n", df.head())

Dataset Shape: (1048575, 11)

Transaction Types Distribution:
 type
CASH_OUT    373641
PAYMENT     353873
CASH_IN     227130
TRANSFER     86753
DEBIT         7178
Name: count, dtype: int64

Processed Shape (Filtered TRANSFER & CASH_OUT): (460394, 10)
Fraud Distribution:
 isFraud
0    459252
1      1142
Name: count, dtype: int64

Head:
     step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
2      1 -0.770970      -0.003443             0.0       -0.359311   
3      1 -0.770970      -0.003443             0.0       -0.345413   
15     1  0.226189       0.457927             0.0       -0.355976   
19     1  0.165983       0.012521             0.0       -0.344597   
24     1  0.585728       0.321137             0.0       -0.355199   

    newbalanceDest  isFraud  errorBalanceOrig  errorBalanceDest  type_TRANSFER  
2        -0.490442        1         -0.626088      1.810000e+04           True  
3        -0.490442        1         -0.626088      2.136300e+06          False  
15  